In [43]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding
import torch.nn as nn
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from transformers import ModernBertConfig
from tqdm import tqdm






In [44]:
df_train = pd.read_csv("scratch/4641_data/train_dataset.csv")
print(df_train["text_length"].max())
df_train = df_train[["text", "sentiment"]]
df_train["sentiment"] = df_train["sentiment"].map({"negative": 0, "neutral": 1, "positive": 2})

df_val = pd.read_csv("scratch/4641_data/val_dataset.csv")
df_val = df_val[["text", "sentiment"]]
df_val["sentiment"] = df_val["sentiment"].map({"negative": 0, "neutral": 1, "positive": 2})

df_test = pd.read_csv("scratch/4641_data/test_dataset.csv")
df_test = df_test[["text", "sentiment", "bucket"]]
df_test["sentiment"] = df_test["sentiment"].map({"negative": 0, "neutral": 1, "positive": 2})


12930


In [45]:
num_classes = 3

In [46]:
class SentimentDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=3000):
        self.bodies = data["text"].values
        self.labels = data["sentiment"].values
        self.buckets = data["bucket"].values if "bucket" in data.columns else None
        self.tokenizer = tokenizer
        self.max_length = max_length
    def __len__(self):
        return len(self.bodies)
    def __getitem__(self, idx):
        encoded = self.tokenizer(str(self.bodies[idx]), truncation=True, padding=False, max_length = self.max_length, return_tensors = "pt")
        return {"input_ids": encoded["input_ids"].squeeze(), "attention_mask": encoded["attention_mask"].squeeze(), "label": torch.tensor(self.labels[idx])}

In [47]:
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
config = ModernBertConfig.from_pretrained("answerdotai/ModernBERT-base", reference_compile=False)
bert_model = AutoModel.from_pretrained("answerdotai/ModernBERT-base", config=config)

In [48]:
train_data = SentimentDataset(df_train, tokenizer)
val_data = SentimentDataset(df_val, tokenizer)
test_data = SentimentDataset(df_test, tokenizer)

In [49]:

collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(train_data, batch_size=16, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_data, batch_size=16, collate_fn=collator)
test_loader = DataLoader(test_data, batch_size=16, collate_fn=collator)

In [50]:
df_train["token_count"] = df_train["text"].apply(lambda x: len(tokenizer.encode(x)))
print(df_train["token_count"].describe())


count    41999.000000
mean        97.454368
std        142.511286
min          3.000000
25%         20.000000
50%         42.000000
75%        109.000000
max       2907.000000
Name: token_count, dtype: float64


In [51]:
print(len(train_data))
print(len(val_data))
print(len(test_data))

41999
6000
12000


In [52]:
class bertSentimentAnalyzer(nn.Module):
    def __init__ (self, num_classes=num_classes, freeze_base=True):
        super().__init__()
        self.base = bert_model
        self.dropout = nn.Dropout(0.3)
        self.linear1 = nn.Linear(self.base.config.hidden_size, num_classes)
        if freeze_base:
            for param in self.base.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask)
        hidden = out.last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        pooled = self.dropout(pooled)
        return self.linear1(pooled)

In [53]:

model = bertSentimentAnalyzer(freeze_base=True)
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)
#optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
num_epochs = 20
training_steps = num_epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1*training_steps),num_training_steps=training_steps)

In [54]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
#print(device)
model.to(device)
print(device)


cuda


In [ ]:

def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for batch in tqdm(loader, desc="Training"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc="Evaluating"):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            total_loss += loss.item()
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy, all_preds, all_labels


best_val_loss = float("inf")
not_improved = 0

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, scheduler, criterion, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

    if val_loss < best_val_loss:
        not_improved = 0
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model1.pt")
        print("Saved best model!")
    elif val_loss >= best_val_loss:
        not_improved += 1
        if not_improved >= 3:
            break


model.load_state_dict(torch.load("best_model1.pt"))
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion, device)
print(f"\nTest Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")

from sklearn.metrics import classification_report
print(classification_report(test_labels, test_preds, target_names=["negative", "neutral", "positive"]))

import pandas as pd
results_df = pd.DataFrame({
    "label": test_labels,
    "pred": test_preds,
    "bucket": df_test["bucket"].values
})

for bucket in ["super_short", "short", "medium", "long"]:
    subset = results_df[results_df["bucket"] == bucket]
    if len(subset) == 0:
        continue
    print(f"\n--- Bucket: {bucket} (n={len(subset)}) ---")
    print(classification_report(subset["label"], subset["pred"],
                                 target_names=["negative", "neutral", "positive"],
                                 zero_division=0))

from sklearn.metrics import classification_report
print(classification_report(test_labels, test_preds, target_names=["negative", "neutral", "positive"]))


Epoch 1/20


Evaluating: 100%|██████████| 375/375 [00:19<00:00, 19.50it/s]


Train Loss: 0.9605 | Train Acc: 0.5312
Val Loss:   0.8159 | Val Acc:   0.6557
Saved best model!

Epoch 2/20


Evaluating: 100%|██████████| 375/375 [00:20<00:00, 18.64it/s]


Train Loss: 0.8317 | Train Acc: 0.6255
Val Loss:   0.7593 | Val Acc:   0.6770
Saved best model!

Epoch 3/20


...

Train Loss: 0.7974 | Train Acc: 0.6465
Val Loss:   0.7371 | Val Acc:   0.6893

Epoch 18/20


Evaluating: 100%|██████████| 375/375 [00:12<00:00, 30.52it/s]


Train Loss: 0.7949 | Train Acc: 0.6492
Val Loss:   0.7359 | Val Acc:   0.6870


Evaluating: 100%|██████████| 750/750 [00:25<00:00, 29.82it/s]



Test Loss: 0.7398 | Test Acc: 0.6808
              precision    recall  f1-score   support

    negative       0.69      0.76      0.72      4050
     neutral       0.66      0.45      0.53      3451
    positive       0.68      0.79      0.73      4499

    accuracy                           0.68     12000
   macro avg       0.68      0.67      0.66     12000
weighted avg       0.68      0.68      0.67     12000


--- Bucket: super_short (n=4000) ---
              precision    recall  f1-score   support

    negative       0.64      0.67      0.66      1184
     neutral       0.61      0.42      0.50      1170
    positive       0.66      0.78      0.72      1646

    accuracy                           0.64      4000
   macro avg       0.64      0.63      0.62      4000
weighted avg       0.64      0.64      0.63      4000


--- Bucket: short (n=4001) ---
              precision    recall  f1-score   support

    negative       0.70      0.76      0.73      1361
     neutral       0.